read data from bronze layer

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.result"
silver_table = f"{catalog_name}.{silver_schema}.result"

- keep only column require for analytics drop url column
- standardise column with snaka case(driverId ==> driver_id, data ---> race_data)

In [0]:
from pyspark.sql import functions as F
result_df = (
    spark
    .table(bronze_table)
    .select(
        F.col("constructorId"),
        F.col("date"),
        F.col("driverId"),
        F.col("grid"),
        F.col("laps"),
        F.col("number"),
        F.col("points"),
        F.col("position"),
        F.col("positionText"),
        F.col("raceName"),
        F.col("round"),
        F.col("season"),
        F.col("status"),
        F.col("ingestion_date"),
        F.col("source_file"))
    .withColumnsRenamed({
        "constructorId":"constructor_id",
        "date":"race_data",
        "driverId":"driver_id",
        "grid":"grid_position",
        "laps":"completed_laps",
        "number":"car_number",
        "position":"final_position",
        "raceName":"race_name",
        "positionText":"final_position_text"})

)


- filter rows where season,round,constructor_id and driver_id is null as these 4 key create one composite key
- remove duplicate 

In [0]:
print("Before removing null and duplicates : ",result_df.count())

result_valid_df = (
    result_df
    .filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("constructor_id").isNotNull() &
        F.col("driver_id").isNotNull())
    .dropDuplicates(["constructor_id","round","season","driver_id"])
)

print("After removing null and duplicates : ",result_valid_df.count())


#### transform values of column race_name to title case

In [0]:
result_final_df = result_valid_df.withColumn("race_name",F.initcap(F.col("race_name")))

#### write data to silver layer

In [0]:
result_final_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

In [0]:
display(spark.table(silver_table))